# Olist E-Commerce - 01. Exploratory Data Analysis (EDA)
## Brazilian E-Commerce Public Dataset by Olist

### Objective
This notebook performs an end-to-end exploratory data audit across all 9 relational entities of the Olist e-commerce database. We examine table dimensions, missing data patterns, data types, duplicate integrity, and basic statistical distributions.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Styling configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Project paths
PROJECT_ROOT = Path('..').resolve()
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_CLEAN = PROJECT_ROOT / 'data' / 'processed' / 'cleaned'

print("Project root:", PROJECT_ROOT)
print("Clean data directory exists:", DATA_CLEAN.exists())


### 1. Load All Datasets & Inspect Dimensions


In [ ]:
datasets = {
    'customers': pd.read_csv(DATA_CLEAN / 'olist_customers_dataset.csv'),
    'geolocation': pd.read_csv(DATA_CLEAN / 'olist_geolocation_dataset.csv'),
    'order_items': pd.read_csv(DATA_CLEAN / 'olist_order_items_dataset.csv'),
    'order_payments': pd.read_csv(DATA_CLEAN / 'olist_order_payments_dataset.csv'),
    'order_reviews': pd.read_csv(DATA_CLEAN / 'olist_order_reviews_dataset.csv'),
    'orders': pd.read_csv(DATA_CLEAN / 'olist_orders_dataset.csv'),
    'products': pd.read_csv(DATA_CLEAN / 'olist_products_dataset.csv'),
    'sellers': pd.read_csv(DATA_CLEAN / 'olist_sellers_dataset.csv'),
    'translation': pd.read_csv(DATA_CLEAN / 'product_category_name_translation.csv')
}

summary = []
for name, df in datasets.items():
    summary.append({
        'Dataset': name,
        'Rows': len(df),
        'Columns': len(df.columns),
        'Nulls': df.isnull().sum().sum(),
        'Memory_MB': round(df.memory_usage(deep=True).sum() / (1024**2), 2)
    })

df_summary = pd.DataFrame(summary).sort_values('Rows', ascending=False)
display(df_summary)


### 2. Inspect Missing Values Across Datasets


In [ ]:
null_breakdown = []
for name, df in datasets.items():
    null_cols = df.isnull().sum()
    null_cols = null_cols[null_cols > 0]
    for col, count in null_cols.items():
        null_breakdown.append({
            'Dataset': name,
            'Column': col,
            'Missing_Count': count,
            'Missing_%': round((count / len(df)) * 100, 2)
        })

df_nulls = pd.DataFrame(null_breakdown)
print("Summary of columns containing missing values:")
display(df_nulls)


### 3. Order Status Distribution
Delivered orders comprise the vast majority (>97%) of transactional volume.


In [ ]:
df_orders = datasets['orders']
status_counts = df_orders['order_status'].value_counts()
status_pct = df_orders['order_status'].value_counts(normalize=True) * 100

order_status_summary = pd.DataFrame({'Orders': status_counts, 'Percentage': status_pct.round(2)})
display(order_status_summary)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=status_counts.values, y=status_counts.index, palette='Blues_r', ax=ax)
ax.set_title('Distribution of Order Statuses', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Orders')
for i, v in enumerate(status_counts.values):
    pct_val = status_pct.iloc[i]
    ax.text(v + 1000, i, f"{v:,} ({pct_val:.2f}%)", va='center')
plt.tight_layout()
plt.show()


### 4. Payment Method Breakdown
Credit card is overwhelmingly the dominant payment vehicle, followed by bank cash vouchers (Boleto).


In [ ]:
df_pay = datasets['order_payments']
pay_summary = df_pay.groupby('payment_type').agg(
    Transactions=('order_id', 'count'),
    Total_Value=('payment_value', 'sum'),
    Avg_Installments=('payment_installments', 'mean')
).reset_index()

pay_summary['Value_%'] = (pay_summary['Total_Value'] / pay_summary['Total_Value'].sum() * 100).round(2)
pay_summary = pay_summary.sort_values('Total_Value', ascending=False)
display(pay_summary)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors = ['#0084FF', '#FF8A00', '#00C49F', '#7952B3', '#AAAAAA']
ax1.pie(pay_summary['Total_Value'], labels=pay_summary['payment_type'], autopct='%1.1f%%',
        startangle=140, colors=colors, textprops={'fontsize': 10})
ax1.set_title('Revenue Share by Payment Type', fontweight='bold')

sns.barplot(data=pay_summary, x='payment_type', y='Avg_Installments', palette='Blues_d', ax=ax2)
ax2.set_title('Average Installments by Payment Type', fontweight='bold')
ax2.set_ylabel('Mean Installments')
plt.tight_layout()
plt.show()


### 5. Review Score Distribution
Over 57% of reviews award 5 stars, leading to a strong 4.09 platform average.


In [ ]:
df_rev = datasets['order_reviews']
score_counts = df_rev['review_score'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(score_counts.index, score_counts.values, color='#0084FF', edgecolor='#071426', width=0.6)
ax.set_title(f'Customer Review Score Distribution (Mean: {df_rev["review_score"].mean():.2f}/5.00)', fontweight='bold')
ax.set_xlabel('Review Score (Stars)')
ax.set_ylabel('Count')
for bar in bars:
    yval = bar.get_height()
    pct = (yval / len(df_rev) * 100)
    ax.text(bar.get_x() + bar.get_width()/2, yval + 1000, f'{yval:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()
